## FINETUNE V1.1 FOR GENERAL DATA

In [ ]:
import sentence_transformers

MODEL_DIR = ''
DB_DIR = ''

In [ ]:
import sqlite3


def get_data(batch=10_000):
    con = sqlite3.connect(DB_DIR)
    con.row_factory = sqlite3.Row

    query = """
        SELECT anchor, positive, hard_negative
        FROM general
        WHERE anchor IS NOT NULL
          AND positive IS NOT NULL
          AND hard_negative IS NOT NULL
        ORDER BY RANDOM()
        LIMIT 1500000
    """
    cursor = con.cursor()
    cursor.execute(query)

    while True:
        rows = cursor.fetchmany(batch)
        if not rows:
            break

        for row in rows:
            yield {
                'anchor': 'query: ' + row['anchor'],
                'positive': 'passage: ' + row['positive'],
                'hard_negative': 'passage: ' + row['hard_negative'],
            }

    cursor.close()
    con.close()

In [ ]:
import math
import torch
from datasets import IterableDataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

model = SentenceTransformer(MODEL_DIR)
loss = MultipleNegativesRankingLoss(model)

train_dataset = IterableDataset.from_generator(get_data)

num_records = 1_500_000
batch_size = 32
epochs = 2
gradient_accumulation_steps = 1

num_gpus = max(torch.cuda.device_count(), 1)
effective_batch_size = batch_size * num_gpus * gradient_accumulation_steps
steps_per_epoch = math.ceil(num_records / effective_batch_size)
max_steps = steps_per_epoch * epochs

print('GPUs:', num_gpus)
print('Effective batch size:', effective_batch_size)
print('Steps / epoch:', steps_per_epoch)
print('Max steps:', max_steps)

args = SentenceTransformerTrainingArguments(
    output_dir='models/vietnamese-embedding-v1.1-general',
    num_train_epochs=epochs,
    max_steps=max_steps,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=100,
    save_strategy='steps',
    save_steps=5000,
    save_total_limit=2,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

In [ ]:
trainer.train()